In [12]:

# 04 — SPECTRAL-TAXONOMIC RELATIONSHIPS
# Spring 2025 only for now

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

#paths to datasets
processed_dir = (
    r"C:\Users\Gabriela Shirkey\OneDrive - Chapman University"
    r"\CONNECT_SBG\Coastal-Tanner\SpectralEvolution"
    r"\LCDM_2005_3A02\LCDM_2005_3A02_processed"
)

taxonomy_dir = (
    r"C:\Users\Gabriela Shirkey\OneDrive - Chapman University"
    r"\CONNECT_SBG\Coastal-Tanner\Maize_IDwork"
)

taxonomy_file = os.path.join(
    taxonomy_dir,
    "FINAL_taxonomic_LCDM_2025_df.csv"
)

spectral_diversity_file = os.path.join(
    processed_dir,
    "LCDM_050225_quadrat_spectral_diversity.csv"
)

crosswalk_file = os.path.join(
    processed_dir,
    "LCDM_050225_filenames_joined_sed_photoids.csv"
)

#Read datasets
taxonomy_df = pd.read_csv(taxonomy_file)
spectral_diversity_df = pd.read_csv(spectral_diversity_file)
crosswalk_df = pd.read_csv(crosswalk_file)

# Spring observations only
spring_taxonomy = taxonomy_df[
    taxonomy_df["date"].astype(str).str.strip() == "5/2/25"
].copy()

print("Full taxonomy:", taxonomy_df.shape)
print("Spring taxonomy:", spring_taxonomy.shape)
print("Spectral diversity:", spectral_diversity_df.shape)
print("Spectral crosswalk:", crosswalk_df.shape)

display(spring_taxonomy.head())

Full taxonomy: (102, 33)
Spring taxonomy: (60, 33)
Spectral diversity: (59, 8)
Spectral crosswalk: (177, 8)


,date,site,file_name,transect_no,quadrat_no,Meter,species,dom_sub,alg_bare,red_corr,...,bro_cylinders,bro_rad_branch,bro_feather_branching,bro_kelp,seagrass,gre_filaments,gre_blade,gre_spongy,alg_unknown,notes
0,5/2/25,LCDM,IMG_2404,1,1,0.0,NaN,bedrock,100.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,5/2/25,LCDM,IMG_2405,1,2,10.0,NaN,sand,100.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,5/2/25,LCDM,IMG_2406,1,3,20.0,NaN,bedrock,98.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,NaN
3,5/2/25,LCDM,IMG_2407,1,4,30.0,NaN,bedrock,99.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN
4,5/2/25,LCDM,IMG_2408,1,5,40.0,NaN,bedrock,33.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,67.0,NaN,14.0,NaN


In [13]:

# VERIFY PHOTO-ID MATCHING BETWEEN SPECTRA AND TAXONOMY
# Not all photos were ID'd because we saw many were just rock. So there will be lots of mis=matches

# Collapse spectral crosswalk from 177 spectra
# to one record per sampled quadrat
quadrat_crosswalk = (
    crosswalk_df[
        [
            "photo_id",
            "quadrat_id",
            "transect",
            "distance_m"
        ]
    ]
    .drop_duplicates()
    .copy()
)

print("Unique spectral quadrats:", len(quadrat_crosswalk))

# Keep spring taxonomy rows that have photo IDs
spring_assessed = spring_taxonomy[
    spring_taxonomy["file_name"].notna()
].copy()

print("Spring rows with taxonomy photo IDs:", len(spring_assessed))


# STANDARDIZE PHOTO IDs
# Remove whitespace and image extensions so:
# IMG_2404.JPG -> IMG_2404
quadrat_crosswalk["photo_id_clean"] = (
    quadrat_crosswalk["photo_id"]
    .astype(str)
    .str.strip()
    .str.replace(r"\.(JPG|JPEG)$", "", regex=True, case=False)
)

spring_assessed["photo_id_clean"] = (
    spring_assessed["file_name"]
    .astype(str)
    .str.strip()
    .str.replace(r"\.(JPG|JPEG)$", "", regex=True, case=False)
)


# CHECK MATCHES
spectral_photos = set(quadrat_crosswalk["photo_id_clean"])
taxonomy_photos = set(spring_assessed["photo_id_clean"])

matched_photos = spectral_photos & taxonomy_photos
spectral_only = spectral_photos - taxonomy_photos
taxonomy_only = taxonomy_photos - spectral_photos

print("\nMatched photo IDs:", len(matched_photos))
print("Spectral photos without taxonomy:", len(spectral_only))
print("Taxonomy photos without spectra:", len(taxonomy_only))

print("\nSpectral-only photo IDs:")
print(sorted(spectral_only))

print("\nTaxonomy-only photo IDs:")
print(sorted(taxonomy_only))

Unique spectral quadrats: 59
Spring rows with taxonomy photo IDs: 42

Matched photo IDs: 42
Spectral photos without taxonomy: 17
Taxonomy photos without spectra: 0

Spectral-only photo IDs:
['IMG_2454', 'IMG_2457', 'IMG_2458', 'IMG_2459', 'IMG_2460', 'IMG_2461', 'IMG_2462', 'IMG_2463', 'IMG_2464', 'IMG_2466', 'IMG_2467', 'IMG_2468', 'IMG_2469', 'IMG_2471', 'IMG_2472', 'IMG_2473', 'IMG_2474']

Taxonomy-only photo IDs:
[]


All 42 taxonomy photo IDs match a spectral quadrat, with zero taxonomy-only records. The remaining 17 are simply the spectral quadrats without taxonomy entered yet.

Richness is defined as the count of unique 'sub categories' listed under richness_columns below. We do not have a definitive list of species, but grouped by likeness, which can reflect physiological and functional traits. 

In [14]:
# Morphological / functional biological cover categories
# alg_bare and alg_unknown are intentionally excluded
richness_columns = [
    "red_corr",
    "red_soft_crusts",
    "red_filaments",
    "red_cylinders",
    "red_bumpy_blades",
    "red_ribs_veins",
    "red_simple_blades",
    "red_branched_single_plane",
    "red_bushy_branched",
    "bro_crusts",
    "bro_filaments",
    "bro_globular",
    "bro_small_flat_blades",
    "bro_dich_branched",
    "bro_cylinders",
    "bro_rad_branch",
    "bro_feather_branching",
    "bro_kelp",
    "seagrass",
    "gre_filaments",
    "gre_blade",
    "gre_spongy"
]

# Make sure cover columns are numeric
spring_assessed[richness_columns] = (
    spring_assessed[richness_columns]
    .apply(pd.to_numeric, errors="coerce")
)

# Count the number of categories with >1% cover
spring_assessed["category_richness"] = (
    spring_assessed[richness_columns]
    .gt(1)
    .sum(axis=1)
)

# Inspect
display(
    spring_assessed[
        [
            "file_name",
            "transect_no",
            "quadrat_no",
            "Meter",
            "category_richness"
        ]
    ]
)

print("\nCategory richness summary:")
print(spring_assessed["category_richness"].describe())

print("\nCategory richness frequency:")
print(
    spring_assessed["category_richness"]
    .value_counts()
    .sort_index()
)

,file_name,transect_no,quadrat_no,Meter,category_richness
0,IMG_2404,1,1,0.0,0
1,IMG_2405,1,2,10.0,0
2,IMG_2406,1,3,20.0,0
3,IMG_2407,1,4,30.0,0
4,IMG_2408,1,5,40.0,1
5,IMG_2409,1,6,50.0,2
6,IMG_2411,2,7,0.0,1
7,IMG_2412,2,8,10.0,0
8,IMG_2413,2,9,20.0,0
9,IMG_2414,2,10,30.0,0



Category richness summary:
count    42.000000
mean      1.142857
std       1.539265
min       0.000000
25%       0.000000
50%       1.000000
75%       2.000000
max       6.000000
Name: category_richness, dtype: float64

Category richness frequency:
category_richness
0    20
1    10
2     5
3     4
5     2
6     1
Name: count, dtype: int64


Percent coverage is defined as the percent total vegetation cover. We subtract the estimated % bare coverage from 100%.

In [17]:
# Convert bare cover to numeric
spring_assessed["alg_bare"] = pd.to_numeric(
    spring_assessed["alg_bare"],
    errors="coerce"
)

# Exclude IMG_2439:
# 93% of the quadrat could not be confidently characterized
# because water/bubbles obscured the surface
spring_assessed = spring_assessed[
    spring_assessed["file_name"] != "IMG_2439"
].copy()


# ---------------------------------------------------------
# Recover missing bare-cover values where possible
# ---------------------------------------------------------

# Biological cover categories, including unknown algae
cover_columns = richness_columns + ["alg_unknown"]

spring_assessed[cover_columns] = spring_assessed[
    cover_columns
].apply(pd.to_numeric, errors="coerce")

# Total recorded biological cover
spring_assessed["cover_sum"] = (
    spring_assessed[cover_columns]
    .fillna(0)
    .sum(axis=1)
)

# If alg_bare is missing AND biological cover sums to 100%,
# the quadrat is interpreted as 0% bare
recoverable = (
    spring_assessed["alg_bare"].isna()
    & np.isclose(spring_assessed["cover_sum"], 100)
)

spring_assessed.loc[recoverable, "alg_bare"] = 0


# ---------------------------------------------------------
# Calculate vegetation cover
# ---------------------------------------------------------

spring_assessed["veg_cover"] = (
    100 - spring_assessed["alg_bare"]
)


# ---------------------------------------------------------
# Verify
# ---------------------------------------------------------

display(
    spring_assessed[
        [
            "file_name",
            "transect_no",
            "quadrat_no",
            "Meter",
            "alg_bare",
            "veg_cover",
            "category_richness"
        ]
    ]
)

print("\nUsable quadrats:", len(spring_assessed))
print("Missing veg cover:", spring_assessed["veg_cover"].isna().sum())

print("\nVeg cover summary:")
print(spring_assessed["veg_cover"].describe())

,file_name,transect_no,quadrat_no,Meter,alg_bare,veg_cover,category_richness
0,IMG_2404,1,1,0.0,100.0,0.0,0
1,IMG_2405,1,2,10.0,100.0,0.0,0
2,IMG_2406,1,3,20.0,98.0,2.0,0
3,IMG_2407,1,4,30.0,99.0,1.0,0
4,IMG_2408,1,5,40.0,33.0,67.0,1
5,IMG_2409,1,6,50.0,19.0,81.0,2
6,IMG_2411,2,7,0.0,79.0,21.0,1
7,IMG_2412,2,8,10.0,100.0,0.0,0
8,IMG_2413,2,9,20.0,100.0,0.0,0
9,IMG_2414,2,10,30.0,100.0,0.0,0



Usable quadrats: 41
Missing veg cover: 0

Veg cover summary:
count     41.000000
mean      30.195122
std       39.126857
min        0.000000
25%        0.000000
50%        2.000000
75%       67.000000
max      100.000000
Name: veg_cover, dtype: float64


Now join the dataframe with the spectral information. Careful to check that the spectral information on images is preserved over the taxonomic, which will inherit the corrected spectral/spatial info. 

In [18]:
# JOIN TAXONOMIC + SPECTRAL DIVERSITY DATA

# Keep ecological variables from the taxonomy dataset
taxonomy_metrics = spring_assessed[
    [
        "photo_id_clean",
        "category_richness",
        "veg_cover"
    ]
].copy()

# Join taxonomy to the corrected quadrat crosswalk using photo ID
analysis_df = quadrat_crosswalk.merge(
    taxonomy_metrics,
    on="photo_id_clean",
    how="inner"
)

# Add spectral diversity metrics using corrected quadrat ID
analysis_df = analysis_df.merge(
    spectral_diversity_df[
        [
            "quadrat_id",
            "mean_spectral_angle_deg",
            "mean_pca_dispersion"
        ]
    ],
    on="quadrat_id",
    how="left"
)

# Keep final analysis variables
analysis_df = analysis_df[
    [
        "photo_id_clean",
        "quadrat_id",
        "transect",
        "distance_m",
        "category_richness",
        "veg_cover",
        "mean_spectral_angle_deg",
        "mean_pca_dispersion"
    ]
].copy()

# Sort spatially
analysis_df = analysis_df.sort_values(
    ["transect", "distance_m"]
).reset_index(drop=True)


# VERIFY
print("Analysis quadrats:", len(analysis_df))

print("\nMissing values:")
print(analysis_df.isna().sum())

display(analysis_df)

Analysis quadrats: 41

Missing values:
photo_id_clean             0
quadrat_id                 0
transect                   0
distance_m                 0
category_richness          0
veg_cover                  0
mean_spectral_angle_deg    0
mean_pca_dispersion        0
dtype: int64


,photo_id_clean,quadrat_id,transect,distance_m,category_richness,veg_cover,mean_spectral_angle_deg,mean_pca_dispersion
0,IMG_2404,1,1,0,0,0.0,1.125146,0.112994
1,IMG_2405,2,1,10,0,0.0,0.425068,0.031022
2,IMG_2406,3,1,20,0,2.0,1.765964,0.028455
3,IMG_2407,4,1,30,0,1.0,0.975065,0.034472
4,IMG_2408,5,1,40,1,67.0,1.457308,0.075512
5,IMG_2409,6,1,50,2,81.0,0.668452,0.039828
6,IMG_2411,7,2,0,1,21.0,2.118695,0.070937
7,IMG_2412,8,2,10,0,0.0,1.252196,0.023023
8,IMG_2413,9,2,20,0,0.0,1.104111,0.020654
9,IMG_2414,10,2,30,0,0.0,5.619797,0.446776


Preliminary Spearman correlations for the abstract
Spearman converts the observations to ranks and asks whether there is a monotonic relationship: as one variable gets larger, does the other generally tend to get larger or smaller? This is better than Pearson's for using extreme values like we have, which would examine the linear relationship. 

In [19]:
from scipy.stats import spearmanr

relationships = [
    ("category_richness", "mean_spectral_angle_deg"),
    ("category_richness", "mean_pca_dispersion"),
    ("veg_cover", "mean_spectral_angle_deg"),
    ("veg_cover", "mean_pca_dispersion")
]

correlation_results = []

for x_var, y_var in relationships:

    rho, p_value = spearmanr(
        analysis_df[x_var],
        analysis_df[y_var]
    )

    correlation_results.append({
        "x_variable": x_var,
        "y_variable": y_var,
        "n": len(analysis_df),
        "spearman_rho": rho,
        "p_value": p_value
    })

correlation_results_df = pd.DataFrame(correlation_results)

display(correlation_results_df.round(4))

,x_variable,y_variable,n,spearman_rho,p_value
0,category_richness,mean_spectral_angle_deg,41,-0.1387,0.3871
1,category_richness,mean_pca_dispersion,41,0.0509,0.7521
2,veg_cover,mean_spectral_angle_deg,41,-0.1110,0.4897
3,veg_cover,mean_pca_dispersion,41,0.0664,0.6798


Ok, so maybe we will need to revise the spectral diversity scale here. Plot-level diversity across 3 samples isn't the way to go... but maybe we want to see diversity between plots. 

In [20]:

# GROUP DIFFERENCES IN SPECTRAL DIVERSITY


from scipy.stats import kruskal
import pandas as pd

results = []

# Test spectral diversity among:
#   1. biological richness classes
#   2. shoreline-distance classes

grouping_variables = [
    "category_richness",
    "distance_m"
]

spectral_variables = [
    "mean_spectral_angle_deg",
    "mean_pca_dispersion"
]

for group_var in grouping_variables:
    
    for spectral_var in spectral_variables:
        
        groups = [
            group[spectral_var].dropna().values
            for _, group in analysis_df.groupby(group_var)
        ]
        
        H, p = kruskal(*groups)
        
        results.append({
            "grouping_variable": group_var,
            "spectral_variable": spectral_var,
            "n": len(analysis_df),
            "kruskal_H": H,
            "p_value": p
        })

kruskal_results_df = pd.DataFrame(results)

display(kruskal_results_df.round(4))

,grouping_variable,spectral_variable,n,kruskal_H,p_value
0,category_richness,mean_spectral_angle_deg,41,3.2249,0.6654
1,category_richness,mean_pca_dispersion,41,2.1146,0.8331
2,distance_m,mean_spectral_angle_deg,41,3.4591,0.6296
3,distance_m,mean_pca_dispersion,41,2.5014,0.7763
